In [28]:
# --- Cell 1: Build focused election dataset file ---
from pathlib import Path
import sys

project_root = Path("/data/disk4/workspace/projects/union")
script_path = project_root / "src" / "build_election_focus_dataset.py"

if not script_path.exists():
    raise FileNotFoundError(f"Script not found: {script_path}")

# Run the focused build script from notebook
exec(script_path.read_text(), {"__name__": "__main__"})
print("\nFocused dataset build completed.")

Reading parquet: /data/disk4/workspace/projects/union/outputs/preliminary_election_level.parquet
Wide shape: (33477, 14107)
Duplicate rows in focus dataset before drop: 0
Focus shape: (33477, 23)
Focus columns: ['election_id', 'case_number', 'election_date', 'tally_type', 'ballot_type', 'runoff_required', 'union_to_certify', 'total_ballots_counted', 'void_ballots', 'challenged_ballots', 'challenges_are_determinative', 'employer_name', 'participant_union_names', 'participant_types_observed', 'unions_involved', 'filing_status', 'filing_date_filed', 'filing_date_closed', 'filing_city', 'filing_state', 'participant_row_count', 'result_row_count', 'tally_row_count']
 election_id  case_number election_date tally_type               ballot_type runoff_required                                  union_to_certify  total_ballots_counted  void_ballots  challenged_ballots challenges_are_determinative                                                                                     employer_name    

In [34]:
# --- Cell 2: Load focused dataset for inspection ---
import pandas as pd
from pathlib import Path

focus_csv = Path("/data/disk4/workspace/projects/union/outputs/preliminary_election_focus.csv")
focus_parquet = Path("/data/disk4/workspace/projects/union/outputs/preliminary_election_focus.parquet")

if focus_parquet.exists():
    df_focus = pd.read_parquet(focus_parquet)
    source_used = str(focus_parquet)
elif focus_csv.exists():
    df_focus = pd.read_csv(focus_csv)
    source_used = str(focus_csv)
else:
    raise FileNotFoundError("Focused output not found. Run Cell 1 first.")

print(f"Loaded from: {source_used}")
print(f"Shape: {df_focus.shape}")
print(f"Columns ({len(df_focus.columns)}):")
print(df_focus.columns.tolist())

Loaded from: /data/disk4/workspace/projects/union/outputs/preliminary_election_focus.parquet
Shape: (33477, 39)
Columns (39):
['election_id', 'case_number', 'election_date', 'tally_type', 'ballot_type', 'runoff_required', 'union_to_certify', 'total_ballots_counted', 'void_ballots', 'challenged_ballots', 'challenges_are_determinative', 'employer_name', 'participant_union_names', 'participant_types_observed', 'unions_involved', 'filing_status', 'filing_date_filed', 'filing_date_closed', 'filing_city', 'filing_state', 'participant_row_count', 'result_row_count', 'tally_row_count', 'votes_for_union', 'votes_against_union', 'total_valid_votes', 'union_support_rate', 'lead_union_name', 'lead_union_votes', 'employer_names_raw', 'employer_types_raw', 'employer_subtypes_raw', 'employer_address_raw', 'employer_address_1_raw', 'employer_address_2_raw', 'employer_city_raw', 'employer_state_raw', 'employer_zip_raw', 'employer_phone_raw']


In [30]:
# --- Cell 3: Quick QA view (result/date/employer/union) ---
key_cols = [
    "election_id",
    "case_number",
    "election_date",
    "tally_type",
    "ballot_type",
    "runoff_required",
    "union_to_certify",
    "unions_involved",
    "employer_name",
    "filing_status",
    "filing_state",
]

existing_key_cols = [c for c in key_cols if c in df_focus.columns]
print("Preview of key columns:")
print(df_focus[existing_key_cols].head(10).to_string(index=False))

print("\nMissingness (key columns):")
print(df_focus[existing_key_cols].isna().mean().sort_values(ascending=False).to_string())

print("\nTop unions_involved:")
if "unions_involved" in df_focus.columns:
    print(df_focus["unions_involved"].value_counts(dropna=False).head(15).to_string())
else:
    print("Column unions_involved not found.")

Preview of key columns:
 election_id  case_number election_date tally_type               ballot_type runoff_required                                                                                     union_to_certify                                                                                                                                                                                                                                                                                                                                                                                                            unions_involved                                                                                                                                                                                       employer_name filing_status filing_state
           1 01-RC-020966    1999-03-17    Initial Single Labor Organization               N                                                        

In [31]:
df_focus.head()

,election_id,case_number,election_date,tally_type,ballot_type,runoff_required,union_to_certify,total_ballots_counted,void_ballots,challenged_ballots,challenges_are_determinative,employer_name,participant_union_names,participant_types_observed,unions_involved,filing_status,filing_date_filed,filing_date_closed,filing_city,filing_state,participant_row_count,result_row_count,tally_row_count
0,1,01-RC-020966,1999-03-17,Initial,Single Labor Organization,N,None,0.0,NaN,7.0,Y,NEW ENGLAND MECHANICAL SERVICES...,INTERNATIONAL BROTHERHOOD OF TE...,Petitioner | Employer,INTERNATIONAL BROTHERHOOD OF TE...,Closed,1999-01-19,1999-08-04,Woburn,MA,2.0,1,2
1,2,01-RC-020986,1999-04-22,Initial,Single Labor Organization,N,SERVICE EMPLOYEES INTERNATIONAL...,148.0,1.0,4.0,N,COURTYARD NURSING CARE CENTER,SERVICE EMPLOYEES INTERNATIONAL...,Not Specified | Employer | Union,SERVICE EMPLOYEES INTERNATIONAL...,Closed,1999-03-12,1999-06-11,Medford,MA,3.0,1,2
2,3,01-RC-021152,2000-04-27,Initial,Single Labor Organization,N,None,424.0,NaN,8.0,N,STEWARD CARITAS CARNEY HOSPITAL,SERVICE EMPLOYEES INTERNATIONAL...,Not Specified | Employer | Union,SERVICE EMPLOYEES INTERNATIONAL...,Closed,2000-03-08,2008-06-03,Boston,MA,7.0,1,2
3,4,01-RC-021561,2002-11-07,Initial,Two Labor Organizations,N,None,99.0,1.0,NaN,N,TELCOM USA INC.,"NEW ENGLAND JOINT BOARD, UNION ...",Not Specified | Employer | Union,"NEW ENGLAND JOINT BOARD, UNION ...",Closed,2002-10-07,2008-07-02,Lawrence,MA,4.0,1,2
4,5,01-RC-022002,2007-04-25,Initial,Single Labor Organization,N,"AMALGAMATED TRANSIT UNION, LOCA...",9.0,NaN,NaN,N,"C. BROOKS & DOUGLAS P. CURRIER,...","VEITCH, FREDERICK ZUCKERMAN, VE...",Petitioner | Employer,"VEITCH, FREDERICK\nZUCKERMAN, V...",Closed,2006-03-24,2007-05-07,Portland,ME,4.0,1,2


In [32]:
# --- Check employer_name multiplicity ---
import pandas as pd

s = df_focus["employer_name"].astype("string")
non_null = s.dropna()

multi_mask = non_null.str.contains("\|", regex=True)
multi_count = int(multi_mask.sum())
single_count = int((~multi_mask).sum())

print(f"non-null employer_name rows: {len(non_null):,}")
print(f"single employer name rows:   {single_count:,}")
print(f"multi employer name rows:    {multi_count:,}")
print(f"multi-name share:            {multi_count / len(non_null):.2%}")

print("\nExamples of multi-employer entries:")
print(non_null[multi_mask].head(10).to_string(index=False))

non-null employer_name rows: 33,446
single employer name rows:   9,792
multi employer name rows:    23,654
multi-name share:            70.72%

Examples of multi-employer entries:
C. BROOKS & DOUGLAS P. CURRIER, ...
P JEWELER, BERNARD
Ogletree, Dea...
BUFFALO JR, HENRY
JACBSON BUFFAL...
ROYALL SMITH, THOMAS
Jackson Lew...
CHANDLER, CAROL
STONEMAN, CHANDL...
BELLO, KENNETH
Bello Welsh LLP |...
HIRSCH, JEFFREY
ROBINSON & COLE ...
HIRSCH, JEFFREY
ROBINSON & COLE ...
HIRSCH, JEFFREY
ROBINSON & COLE ...
ADAMS, AUDREY
x | Durham School ...


<>:7: SyntaxWarning: invalid escape sequence '\|'
<>:7: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_45899/4202515569.py:7: SyntaxWarning: invalid escape sequence '\|'
  multi_mask = non_null.str.contains("\|", regex=True)


In [33]:
# --- Raw participant check: employer names per case_number ---
import pandas as pd

name_col = next((c for c in ['participant','name','participant_name','organization_name','entity_name','full_name'] if c in df_participant.columns), None)
if name_col is None:
    raise RuntimeError(f'No name-like column found in df_participant. columns={df_participant.columns.tolist()}')

x = df_participant[['case_number', 'type', name_col]].copy()
t = x['type'].astype('string').str.lower().fillna('')
employer_mask = t.str.contains('employer|company|business|management', regex=True)

x = x.loc[employer_mask].copy()
x['name_norm'] = x[name_col].astype('string').str.strip().replace('', pd.NA)
x = x.dropna(subset=['case_number', 'name_norm'])

case_counts = x.groupby('case_number')['name_norm'].nunique(dropna=True)

n_cases = int(case_counts.shape[0])
one_name_cases = int((case_counts == 1).sum())
multi_name_cases = int((case_counts > 1).sum())

print(f'name column used: {name_col}')
print(f'cases with employer-like records: {n_cases:,}')
print(f'exactly one employer name:        {one_name_cases:,}')
print(f'more than one employer name:      {multi_name_cases:,}')
print(f'share with >1 employer names:     {multi_name_cases / n_cases:.2%}' if n_cases else 'share with >1 employer names: N/A')

print('\nExample cases with multiple employer names:')
for case_no in case_counts[case_counts > 1].sort_values(ascending=False).head(8).index:
    vals = pd.unique(x.loc[x['case_number'] == case_no, 'name_norm'])[:6]
    print(f'- {case_no}: ' + ' | '.join(map(str, vals)))

name column used: participant
cases with employer-like records: 60,188
exactly one employer name:        26,914
more than one employer name:      33,274
share with >1 employer names:     55.28%

Example cases with multiple employer names:
- 27-RC-288050: Knapp, Amy
Sherman & Howard, LLC | Scully, Patrick
Sherman & Howard, LLC | The Public Interest Network | Work for Progress, Inc. | California Public Interest Research Group, Inc. | United States Public Interest Research Group, Inc.
- 21-RC-321314: Howard, George
Quarles LLP | Betts, J.
Quarles LLP | Planned Parenthood-College Avenue Sarah Weddington Center | Planned Parenthood-Corona Health Center | Planned Parenthood-Escondido Health Center | Planned Parenthood-Euclid Avenue Francis Torbert Center
- 19-RC-340061: Finnegan, Colin
Constangy, Brooks, Smith & Prophete, LLP | Davis, Timothy
Constangy, Brooks, Smith & Prophete, LLP | Service Corporation International | PFS Crematory | Portland Funeral Services | Sunset Crematory
- 04-RC-287

In [35]:
# --- Preview: employer details and vote details ---
preview_cols = [
    'election_id',
    'case_number',
    'election_date',
    'employer_name',
    'employer_city_raw',
    'employer_state_raw',
    'employer_phone_raw',
    'union_to_certify',
    'votes_for_union',
    'votes_against_union',
    'total_valid_votes',
    'union_support_rate',
    'lead_union_name',
    'lead_union_votes',
]
preview_cols = [c for c in preview_cols if c in df_focus.columns]
print(df_focus[preview_cols].head(12).to_string(index=False))

 election_id  case_number election_date                                                                                                                                                                                       employer_name employer_city_raw employer_state_raw                                            employer_phone_raw                                                                                     union_to_certify  votes_for_union  votes_against_union  total_valid_votes  union_support_rate                                                                                      lead_union_name  lead_union_votes
           1 01-RC-020966    1999-03-17                                                                                                                                                               NEW ENGLAND MECHANICAL SERVICES OF MA            Boston                 MA                                                          None                                  

In [36]:
df_focus.head()

,election_id,case_number,election_date,tally_type,ballot_type,runoff_required,union_to_certify,total_ballots_counted,void_ballots,challenged_ballots,challenges_are_determinative,employer_name,participant_union_names,participant_types_observed,unions_involved,filing_status,filing_date_filed,filing_date_closed,filing_city,filing_state,participant_row_count,result_row_count,tally_row_count,votes_for_union,votes_against_union,total_valid_votes,union_support_rate,lead_union_name,lead_union_votes,employer_names_raw,employer_types_raw,employer_subtypes_raw,employer_address_raw,employer_address_1_raw,employer_address_2_raw,employer_city_raw,employer_state_raw,employer_zip_raw,employer_phone_raw
0,1,01-RC-020966,1999-03-17,Initial,Single Labor Organization,N,None,0.0,NaN,7.0,Y,NEW ENGLAND MECHANICAL SERVICES...,INTERNATIONAL BROTHERHOOD OF TE...,Petitioner | Employer,INTERNATIONAL BROTHERHOOD OF TE...,Closed,1999-01-19,1999-08-04,Woburn,MA,2.0,1,2,0.0,0.0,0.0,NaN,afscme local 2960,0.0,NEW ENGLAND MECHANICAL SERVICES...,Employer,Employer,"Boston, MA\n-",Washington St,None,Boston,MA,-,None
1,2,01-RC-020986,1999-04-22,Initial,Single Labor Organization,N,SERVICE EMPLOYEES INTERNATIONAL...,148.0,1.0,4.0,N,COURTYARD NURSING CARE CENTER,SERVICE EMPLOYEES INTERNATIONAL...,Not Specified | Employer | Union,SERVICE EMPLOYEES INTERNATIONAL...,Closed,1999-03-12,1999-06-11,Medford,MA,3.0,1,2,101.0,47.0,148.0,0.682432,service employees international...,101.0,COURTYARD NURSING CARE CENTER,Employer,Employer,"Medford, MA\n02155-1644",200 Governors Ave,None,Medford,MA,02155-1644,(781)391-5400
2,3,01-RC-021152,2000-04-27,Initial,Single Labor Organization,N,None,424.0,NaN,8.0,N,STEWARD CARITAS CARNEY HOSPITAL,SERVICE EMPLOYEES INTERNATIONAL...,Not Specified | Employer | Union,SERVICE EMPLOYEES INTERNATIONAL...,Closed,2000-03-08,2008-06-03,Boston,MA,7.0,1,2,0.0,229.0,229.0,0.000000,afscme local 2960,0.0,STEWARD CARITAS CARNEY HOSPITAL,Employer,Employer,"Dorchester, MA\n02124-5615",2100 Dorchester Ave,None,Dorchester,MA,02124-5615,(617)296-4000
3,4,01-RC-021561,2002-11-07,Initial,Two Labor Organizations,N,None,99.0,1.0,NaN,N,TELCOM USA INC.,"NEW ENGLAND JOINT BOARD, UNION ...",Not Specified | Employer | Union,"NEW ENGLAND JOINT BOARD, UNION ...",Closed,2002-10-07,2008-07-02,Lawrence,MA,4.0,1,2,47.0,52.0,99.0,0.474747,new england joint board union ...,47.0,TELCOM USA INC.,Employer,Employer,"Lawrence, MA\n01843-2227",309 Andover St,None,Lawrence,MA,01843-2227,(978)989-9909
4,5,01-RC-022002,2007-04-25,Initial,Single Labor Organization,N,"AMALGAMATED TRANSIT UNION, LOCA...",9.0,NaN,NaN,N,"C. BROOKS & DOUGLAS P. CURRIER,...","VEITCH, FREDERICK ZUCKERMAN, VE...",Petitioner | Employer,"VEITCH, FREDERICK\nZUCKERMAN, V...",Closed,2006-03-24,2007-05-07,Portland,ME,4.0,1,2,5.0,4.0,9.0,0.555556,amalgamated transit union loca...,5.0,"C. BROOKS & DOUGLAS P. CURRIER,...",Employer,Legal Representative | Employer,"One Portland Square, 10th Floor...",127 Saint John St,None,Portland,ME,04102-3042,(207)774-4000 | (207)774-2666


In [37]:
# --- Election date range + remove elections without vote records ---
from pathlib import Path
import pandas as pd

# 1) Election date range
_date = pd.to_datetime(df_focus['election_date'], errors='coerce')
print(f"Election date range: {_date.min().date()} -> {_date.max().date()}")
print(f"Rows with valid election_date: {_date.notna().sum():,} / {len(df_focus):,}")

# 2) Keep only elections with vote records
# Prefer total_valid_votes if present; fallback to votes_for + votes_against
if 'total_valid_votes' in df_focus.columns:
    valid_vote_mask = pd.to_numeric(df_focus['total_valid_votes'], errors='coerce').fillna(0) > 0
else:
    vf = pd.to_numeric(df_focus.get('votes_for_union', 0), errors='coerce').fillna(0)
    va = pd.to_numeric(df_focus.get('votes_against_union', 0), errors='coerce').fillna(0)
    valid_vote_mask = (vf + va) > 0

df_focus_with_votes = df_focus.loc[valid_vote_mask].copy()

print(f"\nBefore filtering: {len(df_focus):,}")
print(f"After filtering : {len(df_focus_with_votes):,}")
print(f"Removed rows    : {len(df_focus) - len(df_focus_with_votes):,}")

# 3) Export filtered dataset
out_dir = Path('/data/disk4/workspace/projects/union/outputs')
out_dir.mkdir(parents=True, exist_ok=True)

out_csv = out_dir / 'preliminary_election_focus_with_votes.csv'
out_parquet = out_dir / 'preliminary_election_focus_with_votes.parquet'

df_focus_with_votes.to_csv(out_csv, index=False)
try:
    df_focus_with_votes.to_parquet(out_parquet, index=False)
    print(f"Exported: {out_parquet}")
except Exception as e:
    print(f"Parquet export skipped: {e}")

print(f"Exported: {out_csv}")

Election date range: 1994-03-04 -> 2026-04-09
Rows with valid election_date: 33,477 / 33,477

Before filtering: 33,477
After filtering : 31,416
Removed rows    : 2,061
Exported: /data/disk4/workspace/projects/union/outputs/preliminary_election_focus_with_votes.parquet
Exported: /data/disk4/workspace/projects/union/outputs/preliminary_election_focus_with_votes.csv


In [38]:
# --- Count RC elections ---
import pandas as pd

# RC defined by case number pattern: *-RC-*
all_case = df_focus['case_number'].astype('string')
all_rc_mask = all_case.str.contains('-RC-', na=False)

print('All focus dataset:')
print(f"  total rows: {len(df_focus):,}")
print(f"  RC rows   : {int(all_rc_mask.sum()):,}")
print(f"  RC share  : {all_rc_mask.mean():.2%}")

if 'df_focus_with_votes' in globals():
    vote_case = df_focus_with_votes['case_number'].astype('string')
    vote_rc_mask = vote_case.str.contains('-RC-', na=False)
    print('\nFocus dataset with votes only:')
    print(f"  total rows: {len(df_focus_with_votes):,}")
    print(f"  RC rows   : {int(vote_rc_mask.sum()):,}")
    print(f"  RC share  : {vote_rc_mask.mean():.2%}")
else:
    print('\ndf_focus_with_votes not found in kernel. Run the previous filtering cell first if needed.')

All focus dataset:
  total rows: 33,477
  RC rows   : 28,483
  RC share  : 85.08%

Focus dataset with votes only:
  total rows: 31,416
  RC rows   : 26,523
  RC share  : 84.43%


In [39]:
# --- Next Step 1: keep only RC with vote records ---
import pandas as pd

if 'df_focus_with_votes' not in globals():
    if 'total_valid_votes' in df_focus.columns:
        mask_votes = pd.to_numeric(df_focus['total_valid_votes'], errors='coerce').fillna(0) > 0
    else:
        vf = pd.to_numeric(df_focus.get('votes_for_union', 0), errors='coerce').fillna(0)
        va = pd.to_numeric(df_focus.get('votes_against_union', 0), errors='coerce').fillna(0)
        mask_votes = (vf + va) > 0
    df_focus_with_votes = df_focus.loc[mask_votes].copy()

rc_mask = df_focus_with_votes['case_number'].astype('string').str.contains('-RC-', na=False)
df_rc_votes = df_focus_with_votes.loc[rc_mask].copy()

print(f"RC with votes shape: {df_rc_votes.shape}")
print(df_rc_votes[['election_id', 'case_number', 'election_date']].head(5).to_string(index=False))

RC with votes shape: (26523, 39)
 election_id  case_number election_date
           2 01-RC-020986    1999-04-22
           3 01-RC-021152    2000-04-27
           4 01-RC-021561    2002-11-07
           5 01-RC-022002    2007-04-25
           6 01-RC-022034    2006-11-17


In [40]:
# --- Next Step 2: build employer name candidates (primary + auxiliary) ---
import pandas as pd

name_cols = [c for c in ['employer_name', 'employer_names_raw'] if c in df_rc_votes.columns]
if not name_cols:
    raise RuntimeError('No employer name columns found in df_rc_votes')

emp_long = df_rc_votes[['election_id', 'case_number'] + name_cols].copy()
for c in name_cols:
    emp_long[c] = emp_long[c].astype('string')

# Split pipe-joined names into auxiliary candidates
records = []
for _, r in emp_long.iterrows():
    raw_values = []
    for c in name_cols:
        v = r[c]
        if pd.notna(v):
            raw_values.extend([x.strip() for x in str(v).split(' | ') if x and x.strip()])
    seen = set()
    dedup = []
    for x in raw_values:
        if x not in seen:
            seen.add(x)
            dedup.append(x)
    for i, nm in enumerate(dedup):
        records.append({
            'election_id': r['election_id'],
            'case_number': r['case_number'],
            'employer_candidate_name': nm,
            'candidate_rank': i + 1,
            'is_primary_candidate': i == 0,
        })

df_emp_candidates = pd.DataFrame(records)

print(f"Employer candidate rows: {len(df_emp_candidates):,}")
print(f"Unique candidate names: {df_emp_candidates['employer_candidate_name'].nunique():,}")
print(df_emp_candidates.head(10).to_string(index=False))

Employer candidate rows: 58,028
Unique candidate names: 31,366
 election_id  case_number                                            employer_candidate_name  candidate_rank  is_primary_candidate
           2 01-RC-020986                                      COURTYARD NURSING CARE CENTER               1                  True
           3 01-RC-021152                                    STEWARD CARITAS CARNEY HOSPITAL               1                  True
           4 01-RC-021561                                                    TELCOM USA INC.               1                  True
           5 01-RC-022002          C. BROOKS & DOUGLAS P. CURRIER, ROBERT\nVerrill Dana, LLP               1                  True
           5 01-RC-022002                              REGIONAL TRANSPORTATION PROGRAM, INC.               2                 False
           6 01-RC-022034 P JEWELER, BERNARD\nOgletree, Deakins, Nash, Smoak & Stewart, P.C.               1                  True
           6 01-RC-0

In [ ]:
# --- Next Step 3: load Compustat company names from WRDS (schema-robust) ---
import pandas as pd
import wrds

WRDS_USERNAME = "wangyouan"
db = wrds.Connection(wrds_username=WRDS_USERNAME)

# 1) Inspect available columns in comp.names
cols_sql = """
SELECT column_name
FROM information_schema.columns
WHERE table_schema = 'comp' AND table_name = 'names'
ORDER BY ordinal_position
"""
cols_df = db.raw_sql(cols_sql)
avail_cols = cols_df['column_name'].tolist()
print(f"comp.names available columns: {len(avail_cols)}")

# 2) Build dynamic select list
preferred = ['gvkey', 'conm', 'tic', 'cik', 'cusip', 'fic', 'loc', 'naics', 'sic']
select_cols = [c for c in preferred if c in avail_cols]
if 'gvkey' not in select_cols or 'conm' not in select_cols:
    raise RuntimeError(f"Required columns missing in comp.names. available={avail_cols}")

query = f"SELECT {', '.join(select_cols)} FROM comp.names WHERE conm IS NOT NULL"
df_comp = db.raw_sql(query)

# 3) Matching name field
df_comp['comp_name_for_match'] = df_comp['conm'].astype('string')
df_comp = df_comp.dropna(subset=['comp_name_for_match']).drop_duplicates(subset=['gvkey', 'comp_name_for_match']).copy()

print(f"Compustat rows loaded: {len(df_comp):,}")
show_cols = [c for c in ['gvkey', 'comp_name_for_match', 'tic', 'sic', 'naics'] if c in df_comp.columns]
print(df_comp[show_cols].head(10).to_string(index=False))

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
comp.names available columns: 11
Compustat rows loaded: 47,098
 gvkey      comp_name_for_match    tic  sic  naics
001000    A & E PLASTIK PAK INC   AE.2 3089   <NA>
001001  A & M FOOD SERVICES INC  AMFD. 5812    722
001002                 AAI CORP AAIC.1 3825   <NA>
001003    A.A. IMPORTING CO INC   ANTQ 5712 442110
001004                 AAR CORP    AIR 5080 423860
001005    A.B.A. INDUSTRIES INC  ABA.2 3724   <NA>
001006             ABC INDS INC  1145B 2711   <NA>
001007     ABKCO INDUSTRIES INC  4135B 3652 334612
001008 ABM COMPUTER SYSTEMS INC  ABMC. 3577 334119
001009       ABS INDUSTRIES INC ABSI.1 3460   3321


In [ ]:
# --- Next Step 4: fuzzy match employer candidates to Compustat names ---
import re
from difflib import SequenceMatcher
import pandas as pd

try:
    from rapidfuzz import fuzz
    USE_RAPIDFUZZ = True
except Exception:
    USE_RAPIDFUZZ = False


def norm_name(s: str) -> str:
    if s is None:
        return ''
    x = str(s).upper().strip()
    x = re.sub(r'[^A-Z0-9 ]+', ' ', x)
    x = re.sub(r'\s+', ' ', x).strip()
    # remove common legal suffixes for better fuzzy signal
    x = re.sub(r'\b(INC|INCORPORATED|CORP|CORPORATION|CO|COMPANY|LLC|L L C|LTD|LIMITED|PLC|LP|L P)\b', ' ', x)
    x = re.sub(r'\s+', ' ', x).strip()
    return x


def score_pair(a: str, b: str) -> float:
    if USE_RAPIDFUZZ:
        return float(fuzz.token_sort_ratio(a, b))
    return 100.0 * SequenceMatcher(None, a, b).ratio()

# unique query and reference names
q = df_emp_candidates[['employer_candidate_name']].drop_duplicates().copy()
q['q_norm'] = q['employer_candidate_name'].map(norm_name)
q = q[q['q_norm'].str.len() > 0].copy()

# Select only columns that actually exist in df_comp
meta_cols_preferred = ['tic', 'cik', 'cusip', 'fic', 'loc', 'naics', 'sic']
meta_cols = [c for c in meta_cols_preferred if c in df_comp.columns]
ref_cols = ['gvkey', 'comp_name_for_match'] + meta_cols

r = df_comp[ref_cols].copy()
r['r_norm'] = r['comp_name_for_match'].map(norm_name)
r = r[r['r_norm'].str.len() > 0].copy()

# blocking index by first character to keep runtime reasonable
ref_by_first = {}
for idx, val in r['r_norm'].items():
    k = val[0]
    ref_by_first.setdefault(k, []).append(idx)

matches = []
for _, row in q.iterrows():
    qn = row['q_norm']
    first = qn[0]
    cand_idx = ref_by_first.get(first, r.index.tolist())

    # optional length filter
    qlen = len(qn)
    cand = r.loc[cand_idx].copy()
    cand = cand[(cand['r_norm'].str.len() >= max(2, qlen - 8)) & (cand['r_norm'].str.len() <= qlen + 8)]
    if cand.empty:
        cand = r.loc[cand_idx].copy()

    best_score = -1.0
    best_row = None
    for _, rr in cand.iterrows():
        sc = score_pair(qn, rr['r_norm'])
        if sc > best_score:
            best_score = sc
            best_row = rr

    if best_row is not None:
        rec = {
            'employer_candidate_name': row['employer_candidate_name'],
            'match_score': best_score,
            'matched_gvkey': best_row['gvkey'],
            'matched_conm': best_row['comp_name_for_match'],
        }
        for c in meta_cols:
            rec[f'matched_{c}'] = best_row[c]
        matches.append(rec)

df_name_match = pd.DataFrame(matches)
print(f"Name-level matches built: {len(df_name_match):,}")
print(df_name_match.sort_values('match_score', ascending=False).head(10).to_string(index=False))

In [ ]:
# --- Next Step 5: collapse candidate matches to election-level best match ---
import pandas as pd
from pathlib import Path

# Join candidate-level matches back
df_emp_match = df_emp_candidates.merge(df_name_match, on='employer_candidate_name', how='left')

# For each election_id, keep best match score across primary + auxiliary employer names
df_best = (
    df_emp_match.sort_values(['election_id', 'match_score'], ascending=[True, False])
    .drop_duplicates(subset=['election_id'], keep='first')
    .copy()
)

keep_cols = [
    'election_id', 'case_number', 'employer_candidate_name', 'candidate_rank', 'is_primary_candidate',
    'match_score', 'matched_gvkey', 'matched_conm', 'matched_tic', 'matched_cik', 'matched_cusip',
    'matched_fic', 'matched_loc', 'matched_naics', 'matched_sic'
]
keep_cols = [c for c in keep_cols if c in df_best.columns]
df_best = df_best[keep_cols]

# Merge into RC vote sample
df_rc_votes_matched = df_rc_votes.merge(df_best, on=['election_id', 'case_number'], how='left')

print(f"Matched RC sample shape: {df_rc_votes_matched.shape}")
print(f"Rows with gvkey match: {df_rc_votes_matched['matched_gvkey'].notna().sum():,}")
print(f"Rows with score>=90: {(df_rc_votes_matched['match_score'] >= 90).fillna(False).sum():,}")
print(df_rc_votes_matched[['election_id', 'case_number', 'employer_name', 'matched_conm', 'matched_gvkey', 'match_score']].head(12).to_string(index=False))

out_dir = Path('/data/disk4/workspace/projects/union/outputs')
out_dir.mkdir(parents=True, exist_ok=True)
out_csv = out_dir / 'preliminary_election_focus_with_votes_rc_compustat_match.csv'
out_parquet = out_dir / 'preliminary_election_focus_with_votes_rc_compustat_match.parquet'

df_rc_votes_matched.to_csv(out_csv, index=False)
try:
    df_rc_votes_matched.to_parquet(out_parquet, index=False)
except Exception as e:
    print(f"Parquet export skipped: {e}")

print(f"Exported: {out_csv}")
print(f"Exported: {out_parquet}")